In [125]:
#step 1

import re
import urllib.request
from urllib.parse import urlparse
from collections import Counter, defaultdict

from bs4 import BeautifulSoup, Comment
from nltk.corpus import stopwords
from nltk import wordpunct_tokenize


# Configuration

COMMON_NOISE_WORDS = set("""
january debt est dec big than who use jun jan feb mar apr may jul agust dec oct nov sep dec
product continue one two three four five please thanks find helpful week job experience women girl
apology read show eve knowledge benefit appointment street way staff salon discount gift cost thing
world close party love letters rewards offers special close page week dollars voucher gifts vouchers
welcome therefore march nights need name pleasure show sisters thank menu today always time needs
welcome march february april may june jully aguast september october november december day year
month minute second secodns www https
""".split())


SPECIAL_CHARS_RE = re.compile(
    r'[`~!@#$%^&*()_+=\\\[\]{};:"<>?,./\-]'
)


def is_visible_text(element) -> bool:
    """Filter out non-visible elements."""
    if element.parent.name in ['html', 'style', 'script', 'head', '[document]', 'img']:
        return False
    if isinstance(element, Comment):
        return False
    return True

def extract_visible_text_from_html(html: bytes) -> str:
    """Return plain visible text from raw HTML."""
    soup = BeautifulSoup(html, 'lxml')
    texts = soup.find_all(string=True)
    visible_texts = filter(is_visible_text, texts)
    return " ".join(t.strip() for t in visible_texts)


# Teacher's ver.
#def normalize_whitespace(text: str) -> str:
#   """Collapse extra whitespace and blank lines."""
#    lines = (line.strip() for line in text.splitlines())
#   chunks = (phrase.strip() for line in lines for phrase in line.split(" "))
#   return "\n".join(chunk for chunk in chunks if chunk)

def normalize_whitespace(text: str) -> str:
    lines = (line.strip() for line in text.splitlines())
    chunks = (phrase.strip() for line in lines for phrase in line.split(" "))
    return " ".join(chunk for chunk in chunks if chunk)

def fetch_page(url: str):
    """Fetch URL, return (clean_text, soup)."""
    req = urllib.request.Request(url, headers={'User-Agent': 'KeywordScraper/1.0'})
    with urllib.request.urlopen(req) as response:
        html = response.read()

    soup = BeautifulSoup(html, 'lxml')
    raw_text = extract_visible_text_from_html(html)
    clean_text = normalize_whitespace(raw_text)
    return clean_text, soup

url = "https://www.wiley.com/en-ie"
clean_text, soup = fetch_page(url)

In [126]:
from urllib.parse import urlparse

def split_url_host(url: str) -> list:
    """Extract host parts as tokens. Example: https://www.amazon.com -> ['amazon', 'com']"""
    parsed = urlparse(url)
    host = (parsed.netloc or "").lower()

    # remove port (e.g., example.com:8080)
    host = host.split(":")[0]

    # remove leading www.
    if host.startswith("www."):
        host = host[4:]

    parts = [p for p in host.split(".") if p]
    return parts


In [127]:
#Step 2

from nltk.corpus import stopwords
from nltk.tokenize import wordpunct_tokenize

def _calculate_language_scores(text: str) -> dict:
    ratios = {}
    tokens = wordpunct_tokenize(text)
    words = [w.lower() for w in tokens]

    words_set = set(words)
    for lang in stopwords.fileids():
        stopwords_set = set(stopwords.words(lang))
        common_elements = words_set.intersection(stopwords_set)
        ratios[lang] = len(common_elements)

    return ratios


def detect_language_and_stopwords(text: str):
    ratios = _calculate_language_scores(text)
    detected_lang = max(ratios, key=ratios.get)
    return detected_lang, set(stopwords.words(detected_lang))



In [128]:
#tokanization

def clean_text_to_words(text: str, stopword_list: set) -> list:
    words = []

    for raw_word in text.split():
        word = raw_word.replace("Â", "").lower()
        word = SPECIAL_CHARS_RE.sub(" ", word).strip()

        if not word:
            continue

        for token in word.split():
            token = token.strip()
            if (
                len(token) > 1
                and not token[0].isdigit()
                and token not in stopword_list
                and token not in COMMON_NOISE_WORDS
                and not token.isdigit()
            ):
                words.append(token)
    return words

clean_text, soup = fetch_page("https://www.wiley.com/en-ie")
detected_lang, stopwrds = detect_language_and_stopwords(clean_text)
words = clean_text_to_words(clean_text, stopwrds)
words


['home',
 'wiley',
 'account',
 'apps',
 'arrow',
 'arrow',
 'left',
 'arrow',
 'arrow',
 'bookmark',
 'filled',
 'bookmark',
 'calendar',
 'check',
 'download',
 'edit',
 'error',
 'expand',
 'filter',
 'grid',
 'view',
 'group',
 'hamburger',
 'heart',
 'history',
 'info',
 'locked',
 'long',
 'arrow',
 'mail',
 'minimize',
 'minus',
 'open',
 'play',
 'print',
 'refresh',
 'schedule',
 'search',
 'share',
 'shopping',
 'bag',
 'square',
 'facebook',
 'square',
 'instagram',
 'square',
 'linkedin',
 'square',
 'twitter',
 'square',
 'twitter',
 'square',
 'youtube',
 'star',
 'dots',
 'thumb',
 'thumb',
 'trash',
 'unlocked',
 'video',
 'visibility',
 'visibility',
 'publish',
 'work',
 'research',
 'teach',
 'grow',
 'solutions',
 'partnerships',
 'insights',
 'shop',
 'future',
 'trusted',
 'research',
 'don’t',
 'miss',
 'industry',
 'experts',
 'discussing',
 'scholarly',
 'societies',
 'innovating',
 'thriving',
 'episode',
 'conversations',
 'watch',
 'full',
 'episode',
 'conv

In [155]:
#Step3
# Three example URL
# URL = "https://www.uef.fi/web/machine-learning/software/deep-learning-tools?version=latest&platform=linux"
#URL = "https://www.bbc.com/news/world-europe/artificial-intelligence-regulation-2025?utm_source=homepage"
URL = "https://www.mozilla.org/en-US"

parsed = urlparse(URL)
host = parsed.hostname or ""

parts = []
for chunk in host.split('.'):
    chunk = chunk.lower()
    if chunk not in ['', 'https', 'www', 'com', '-', 'php', 'pk', 'fi', 'http']:
        parts.append(chunk)
parts


['mozilla', 'org']

In [158]:
from urllib.parse import urlparse

URL = "https://www.wiley.com/en-ie"
host_parts = ["wiley", "com", "en", "ie"]  # 例：split_url_host の結果を想定

path_tokens = []

for segment in URL.split('/'):
    for dot_part in segment.split('.'):
        for dash_part in dot_part.split('-'):
            token = dash_part.lower()
            if (
                token
                and token not in ['https', 'www', 'com', '-', 'php', 'pk', 'fi', 'https:', 'http']
                and token not in host_parts
            ):
                path_tokens.append(token)

print("Path tokens:")
print(path_tokens)


Path tokens:
[]


In [157]:
def split_url_path_and_query(url: str, host_parts: list) -> list:
    """
    Extract path/query segments as tokens, excluding host parts and generic pieces.
    """
    path_tokens = []
    for segment in url.split('/'):
        for dot_part in segment.split('.'):
            for dash_part in dot_part.split('-'):
                token = dash_part.lower()
                if (
                    token
                    and token not in ['https', 'www', 'com', '-', 'php', 'pk', 'fi', 'https:', 'http']
                    and token not in host_parts
                ):
                    path_tokens.append(token)
    
    print(path_tokens)
    return path_tokens



In [131]:
def extract_path_tokens(URL, parts):
    path_tokens = []

    for segment in URL.split('/'):
        for dot_part in segment.split('.'):
            for dash_part in dot_part.split('-'):
                token = dash_part.lower()
                if (
                    token
                    and token not in ['https', 'www', 'com', '-', 'php', 'pk', 'fi', 'http', 'https:']
                    and token not in parts
                ):
                    path_tokens.append(token)

    return path_tokens


In [132]:
#header/anchor/title extraction
def extract_tag_texts(soup, tag_name: str) -> list:
    """Return list of lower-cased texts for given HTML tag."""
    result = []
    for element in soup.find_all(tag_name):
        text = element.get_text(strip=True).lower()
        if text:
            result.append(text)
    return result


In [133]:
def explode_text_dict_to_words(tag_text_dict: dict) -> list:
    words = []
    for _, text_list in tag_text_dict.items():
        for text in text_list:
            for comma_chunk in text.split(','):
                for word in comma_chunk.split():
                    words.append(word)
    return words


In [134]:
def extract_headers_anchors_title_words(soup):
    """Return token lists for H1–H6, <a>, <title>."""
    h1_words = explode_text_dict_to_words({'h1': extract_tag_texts(soup, 'h1')})
    h2_words = explode_text_dict_to_words({'h2': extract_tag_texts(soup, 'h2')})
    h3_words = explode_text_dict_to_words({'h3': extract_tag_texts(soup, 'h3')})
    h4_words = explode_text_dict_to_words({'h4': extract_tag_texts(soup, 'h4')})
    h5_words = explode_text_dict_to_words({'h5': extract_tag_texts(soup, 'h5')})
    h6_words = explode_text_dict_to_words({'h6': extract_tag_texts(soup, 'h6')})
    anchor_words = explode_text_dict_to_words({'a': extract_tag_texts(soup, 'a')})
    title_words = explode_text_dict_to_words({'title': extract_tag_texts(soup, 'title')})

    return (
        h1_words,
        h2_words,
        h3_words,
        h4_words,
        h5_words,
        h6_words,
        anchor_words,
        title_words,
    )

h1_words = extract_headers_anchors_title_words(soup)

In [135]:
#TF

def tf_score(freq: int, total_tokens: int) -> float:
    """
    Your original TF function, kept as-is logically.
    """
    if total_tokens < 50:
        return (freq / 100.0) * 50
    else:
        return (freq / 100.0) * 20


In [136]:
def compute_keyword_scores(words: list, soup, url: str) -> dict:
    freq = Counter(words)
    total_tokens = len(words)

    # headers / anchors / title
    h1, h2, h3, h4, h5, h6, anchor, title = extract_headers_anchors_title_words(soup)

    # URL parts
    url_host = split_url_host(url)
    url_path = split_url_path_and_query(url, url_host)

    headers_names = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6', 'A', 'Title', 'URL-H', 'URL-Q']
    headers_scores = [6, 5, 4, 3, 2, 2, 1, 5, 5, 4]

    headers_lists = [h1, h2, h3, h4, h5, h6, anchor, title, url_host, url_path]

    word_info = {}  # word -> (frequency, [tags], final_score)

    for word, count in freq.items():
        base_tf = tf_score(count, total_tokens)
        tag_scores = []
        tag_names = []

        for idx, tokens in enumerate(headers_lists):
            if word in tokens:
                tag_scores.append(headers_scores[idx])
                tag_names.append(headers_names[idx])

        final_score = base_tf + sum(tag_scores)   # ← これだけ
        word_info[word] = (count, tag_names, final_score)

    return word_info


In [140]:
from collections import Counter

def top_bigram(words: list) -> tuple[tuple[str, str] | None, int]:
    if len(words) < 2:
        return None, 0
    bigrams = Counter(zip(words, words[1:]))
    return bigrams.most_common(1)[0]

In [141]:
#CALL THE MAIN

if __name__ == "__main__":
    URL = "https://www.wiley.com/en-ie"

    text, soup = fetch_page(URL)
    lang, stopword_list = detect_language_and_stopwords(text)
    print("split_url_host" in globals())

    tokens = clean_text_to_words(text, stopword_list)
    keyword_data = compute_keyword_scores(tokens, soup, URL)
    
    # Sort by final_score (3rd element of tuple) descending, take top 10
    top_10 = sorted(
        keyword_data.items(),
        key=lambda kv: kv[1][2],  # kv[1] = (freq, tags, score); [2] = score
        reverse=True
    )[:10]

    print(f"Detected language: {lang}")
    print("Top 10 keywords:")
    for word, (freq, tags, score) in top_10:
        print(f"{word:20} freq={freq:3d} score={score:6.2f} tags={tags}")
    
    bg, cnt = top_bigram(tokens)
    print("Top bigram:", bg, "count=", cnt)

True
Detected language: hinglish
Top 10 keywords:
research             freq= 24 score= 18.80 tags=['H1', 'H3', 'H4', 'A']
wiley                freq= 17 score= 18.40 tags=['H3', 'A', 'Title', 'URL-H']
solutions            freq= 21 score= 12.20 tags=['H3', 'H4', 'A']
shop                 freq=  9 score= 10.80 tags=['H2', 'H4', 'A']
future               freq=  2 score= 10.40 tags=['H1', 'H3']
publish              freq= 11 score= 10.20 tags=['H3', 'H4', 'A']
account              freq=  3 score=  9.60 tags=['H4', 'A', 'Title']
expand               freq=  2 score=  9.40 tags=['H3', 'Title']
work                 freq=  7 score=  9.40 tags=['H3', 'H4', 'A']
societies            freq=  6 score=  9.20 tags=['H3', 'H4', 'A']
Top bigram: ('ai', 'research') count= 4
